# Cropchain Phase 2 — Scrum Planning
## Experiment 1: Scrum Setup, Roles, and Sprint Planning (Team Lead Notebook)

**Project:** Cropchain: Intelligent Farm-to-Table Supply Chain Management System  
**Methodology:** Phase 2 — Agile Scrum (iterative development)  
**Author:** Team Lead  
**Course:** CS587 — Software Project Management  

---

### Purpose of This Notebook

This notebook is the **team lead's contribution** to Phase 2 of the Cropchain final project.  
Phase 2 repeats and **refines** the planning workflow from Phase 1 (Waterfall), but now applies the **Scrum agile framework** to the same Cropchain platform.

The team lead's scope for Phase 2 covers:
1. Shared environment and model setup (reused and validated)
2. Scrum role definitions and AI agent mapping
3. Product Backlog generation from Phase 1 requirements
4. Sprint 1 planning — User Stories, Story Points, and Sprint Goals
5. Scrum Master and Product Owner agent coordination
6. Saving baseline outputs for teammate reuse in Experiments 2 and 3

---

### How Phase 2 Differs from Phase 1

| Dimension | Phase 1 (Waterfall) | Phase 2 (Scrum) |
|---|---|---|
| Structure | Sequential, phase-gated | Iterative, sprint-based |
| Planning unit | Phases and tasks | Sprints and user stories |
| Roles | PM, RE, SE, Dev, Test, Docs | Scrum Master, Product Owner, Dev Team |
| Output | Requirements → Design → Impl → Test → Docs | Product Backlog → Sprint Backlog → Increment |
| Review mechanism | Phase reviews and rework | Sprint Reviews and Retrospectives |
| Estimation unit | Hours / Pages / Feature Points | Story Points |

---

### Phase 1 Issues Identified and Corrected in Phase 2

The following issues were noted in Phase 1 and are corrected here:

1. **Project Manager agent was initially commented out** in Experiment 1 (cell In[1]) — in Phase 2, all agents are active from the start.
2. **Effort estimates were inconsistent** across agents (PM hours did not match RE days) — Phase 2 uses a unified Story Point system.
3. **SLOC was incorrectly used** in Software Engineer estimation before being corrected to Feature Points late in Phase 1 — Phase 2 uses Story Points consistently from the start.
4. **API key warning** (`not a valid OpenAI format`) appeared in every agent instantiation — this is expected when using a non-OpenAI-hosted key (e.g., Azure OpenAI or a proxy); it does not break execution.
5. **Output directory referenced a hardcoded personal path** — Phase 2 uses dynamic `Path.cwd()` resolution consistently.
6. **No Scrum-specific roles were defined in Phase 1** — Phase 2 introduces Scrum Master, Product Owner, and the full Scrum development team.

---

### Scrum Reference Sources (per project instructions)
- https://www.scrum.org/resources/what-scrum-module
- https://www.atlassian.com/agile/scrum/roles

## Section 1: Environment Setup

The environment configuration is shared across all Phase 2 notebooks.  
All teammates must use the same model, API key structure, and output directory conventions.

**Output directory for this experiment:** `src/outputs/phase2/experiment_1/`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import autogen

# ── Directory resolution ──────────────────────────────────────────────────────
CURRENT_DIR = Path.cwd()
BASE_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

# ── Load environment variables ────────────────────────────────────────────────
load_dotenv(BASE_DIR / ".env", override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file.")

# ── Shared model configuration ────────────────────────────────────────────────
MODEL_NAME = "gpt-4o-mini"
TEAM_LLM_CONFIG = {
    "config_list": [
        {
            "model": MODEL_NAME,
            "api_key": OPENAI_API_KEY,
        }
    ],
    "temperature": 0.2,
    "timeout": 120,
}

# ── Project identity ──────────────────────────────────────────────────────────
PROJECT_NAME = "Cropchain: Intelligent Farm-to-Table Supply Chain Management System"

# ── Output directory for Phase 2, Experiment 1 ───────────────────────────────
OUTPUT_DIR = BASE_DIR / "src" / "outputs" / "phase2" / "experiment_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Phase 1 artifacts (reused as context for Scrum planning) ─────────────────
PHASE1_EXP1_DIR = BASE_DIR / "src" / "outputs" / "experiment_1"
PHASE1_REQ_FILE = PHASE1_EXP1_DIR / "04_requirements_engineer_output.md"
PHASE1_PM_FILE  = PHASE1_EXP1_DIR / "03_project_manager_summary.md"

# Load Phase 1 requirements as context for backlog generation
phase1_requirements_text = ""
if PHASE1_REQ_FILE.exists():
    phase1_requirements_text = PHASE1_REQ_FILE.read_text(encoding="utf-8")
    print("Phase 1 requirements loaded for Scrum backlog seeding.")
else:
    print("WARNING: Phase 1 requirements file not found. Backlog will be generated from scratch.")

print("Environment loaded successfully.")
print("Model:", MODEL_NAME)
print("Base directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)
print("API key loaded:", bool(OPENAI_API_KEY))

## Section 2: Cropchain Problem Statement

The customer problem statement is **unchanged from Phase 1**.  
In Phase 2, this same statement is fed into the Scrum workflow instead of the Waterfall workflow.  
The Product Owner agent will interpret this statement and produce a prioritized Product Backlog.

In [ ]:
CUSTOMER_MESSAGE = """
I want to create a web-based platform called Cropchain for the food services domain.
The platform should connect farmers, restaurants, and grocery stores so they can
directly buy and sell crops more efficiently.

The system should support:
1. Farmer onboarding and crop listing
2. Restaurant and grocery buyer accounts
3. Direct crop ordering and contract-based purchasing
4. Crop yield prediction to help estimate future supply
5. Real-time inventory and shipment visibility
6. Fair pricing recommendations based on demand, seasonality, and supply
7. Carbon-aware logistics recommendations for deliveries
8. Freshness, quality, and traceability tracking
9. Alerts for delays, shortages, and spoilage risks
10. Analytics dashboards for supply, demand, and pricing trends

I need a full software project plan using the Scrum agile framework.
""".strip()

print(CUSTOMER_MESSAGE)

## Section 3: Scrum Roles and AI Agent Definitions

In Phase 2, we replace the Waterfall roles with Scrum roles.  
Each role is implemented as an AutoGen `ConversableAgent`.

### Scrum Role Mapping

| Scrum Role | Agent Name | Responsibility |
|---|---|---|
| Customer / Stakeholder | `Customer_Proxy` | Delivers the problem statement |
| Product Owner | `Product_Owner` | Owns and prioritizes the Product Backlog |
| Scrum Master | `Scrum_Master` | Facilitates Sprint planning, removes impediments |
| Developer (Full-Stack) | `Developer_Agent` | Estimates and implements user stories |
| QA / Tester | `QA_Engineer` | Defines acceptance criteria and test coverage |
| UI/UX Designer | `Designer_Agent` | Contributes UI/UX story details |
| DevOps Engineer | `DevOps_Agent` | Handles CI/CD, infrastructure, and deployment stories |
| Technical Writer | `TechWriter_Agent` | Documents Sprint increments |

> **Note:** Per the Phase 2 instructions, the Scrum team includes Developers, Testers/QA, Designers,  
> Business Analysts, DevOps Engineers, Architects, Scrum Master, Security Engineers, and Technical Writers.  
> For this experiment (Team Lead scope), we activate the **Product Owner** and **Scrum Master** agents.  
> Teammates will activate the remaining agents in their respective notebooks.

In [ ]:
# ── Customer Proxy (unchanged from Phase 1) ───────────────────────────────────
customer_proxy_agent = autogen.ConversableAgent(
    name="Customer_Proxy",
    system_message="""
You represent the customer and business stakeholder for Cropchain.
Your job is to clearly describe the business need, users, pain points, and desired features.
Stay focused on business goals, expected benefits, and real-world use cases.
When asked by the Scrum team, you can clarify requirements and prioritize features.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

# ── Product Owner ─────────────────────────────────────────────────────────────
product_owner_agent = autogen.ConversableAgent(
    name="Product_Owner",
    system_message="""
You are the Product Owner for the Cropchain Scrum project.
Your responsibilities:
1. Translate the customer request into a prioritized Product Backlog.
2. Write user stories in the format: 'As a [actor], I want to [action] so that [benefit].'
3. Assign a unique ID to each story: US-001, US-002, etc.
4. Assign a priority to each story: High / Medium / Low.
5. Assign Story Points using the Fibonacci scale: 1, 2, 3, 5, 8, 13.
6. Define clear acceptance criteria for each story.
7. Identify the first Sprint's scope (Sprint 1 Backlog) — the highest-priority stories that fit in a 2-week sprint.
8. Include at least 15 user stories in the Product Backlog.
9. Reflect all Cropchain-specific features: yield prediction, fair pricing, traceability, alerts, logistics visibility.
10. Include actors: Farmer, Restaurant Buyer, Grocery Buyer, Logistics Coordinator, Platform Admin.

Output must include:
A. Full Product Backlog (all user stories)
B. Sprint 1 Backlog (selected subset for first sprint)
C. Sprint Goal for Sprint 1
D. Total Story Points in backlog
E. Story Points committed for Sprint 1
F. Assumptions and open questions

Make the output structured, specific to Cropchain, and presentation-ready.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

# ── Scrum Master ──────────────────────────────────────────────────────────────
scrum_master_agent = autogen.ConversableAgent(
    name="Scrum_Master",
    system_message="""
You are the Scrum Master for the Cropchain Scrum project.
Your responsibilities:
1. Review the Product Backlog produced by the Product Owner.
2. Facilitate Sprint 1 planning by confirming team capacity and sprint commitments.
3. Define Sprint ceremonies: Sprint Planning, Daily Scrum, Sprint Review, Sprint Retrospective.
4. Identify and document impediments, risks, and dependencies.
5. Produce a Sprint 1 Plan that includes:
   - Sprint duration (2 weeks)
   - Team capacity in Story Points per sprint
   - Sprint 1 Backlog items with assignments
   - Sprint ceremonies schedule
   - Definition of Done (DoD)
   - Identified risks and impediments
   - Sprint 1 success criteria
6. Keep the plan aligned with the Cropchain domain and the Product Owner's backlog.
7. Do NOT redefine requirements — work with what the Product Owner has produced.

Make the output structured and presentation-ready.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

# ── Placeholder agents (defined here for project-wide consistency; ─────────────
# ── activated by teammates in their notebooks) ────────────────────────────────
developer_agent = autogen.ConversableAgent(
    name="Developer_Agent",
    system_message="""
You are a Full-Stack Developer on the Cropchain Scrum team.
Your future job: break down user stories into technical tasks, estimate implementation effort,
and define technical dependencies. Use Story Points for estimation.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

qa_engineer_agent = autogen.ConversableAgent(
    name="QA_Engineer",
    system_message="""
You are the QA / Test Engineer on the Cropchain Scrum team.
Your future job: define acceptance criteria, create test cases per user story,
and validate sprint increments against the Definition of Done.
Use 2 test cases per story per day as productivity baseline.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

designer_agent = autogen.ConversableAgent(
    name="Designer_Agent",
    system_message="""
You are the UI/UX Designer on the Cropchain Scrum team.
Your future job: contribute UI/UX design details to user stories,
create wireframe descriptions, and ensure accessibility and usability.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

devops_agent = autogen.ConversableAgent(
    name="DevOps_Agent",
    system_message="""
You are the DevOps Engineer on the Cropchain Scrum team.
Your future job: plan CI/CD pipelines, containerization, cloud deployment,
and infrastructure-as-code for the Cropchain platform.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

tech_writer_agent = autogen.ConversableAgent(
    name="TechWriter_Agent",
    system_message="""
You are the Technical Writer on the Cropchain Scrum team.
Your future job: document each sprint increment, write release notes,
and maintain user-facing and API documentation across sprints.
Use 3 pages per day as productivity baseline.
""".strip(),
    llm_config=TEAM_LLM_CONFIG,
    code_execution_config=False,
    human_input_mode="NEVER",
)

print("All Phase 2 Scrum agents created successfully.")

## Section 4: Prompt Design for Phase 2 Experiment 1

### Step 1 — Customer Proxy → Product Owner
The customer delivers the Cropchain problem statement to the Product Owner, who transforms it into a prioritized Product Backlog with user stories and Story Points.

### Step 2 — Product Owner → Scrum Master
The Product Owner's backlog is handed to the Scrum Master, who produces the Sprint 1 Plan including capacity, ceremonies, Definition of Done, and impediments.

**Key improvements over Phase 1 prompts:**
- Explicit Fibonacci Story Point scale
- Mandatory acceptance criteria per story
- Phase 1 requirements injected as context to ensure continuity
- Sprint ceremony schedule explicitly required
- Definition of Done explicitly required

In [ ]:
# ── Step 1 prompt: Customer → Product Owner ───────────────────────────────────
customer_to_po_prompt = f"""
I want to create a web-based platform called Cropchain for the food services domain.
The platform should connect farmers, restaurants, and grocery stores so they can
directly buy and sell crops more efficiently.

The system should support:
1. Farmer onboarding and crop listing
2. Restaurant and grocery buyer accounts
3. Direct crop ordering and contract-based purchasing
4. Crop yield prediction to help estimate future supply
5. Real-time inventory and shipment visibility
6. Fair pricing recommendations based on demand, seasonality, and supply
7. Carbon-aware logistics recommendations for deliveries
8. Freshness, quality, and traceability tracking
9. Alerts for delays, shortages, and spoilage risks
10. Analytics dashboards for supply, demand, and pricing trends

I need a full software project plan using the Scrum agile framework.

=== CONTEXT FROM PHASE 1 REQUIREMENTS ===
{phase1_requirements_text if phase1_requirements_text else 'Use the customer request above to derive all stories.'}

Using the customer request and the Phase 1 requirements as context, generate a complete
Product Backlog for Cropchain following Scrum practices.

Your output must include:
1. Full Product Backlog — at least 15 user stories
   - Each story ID: US-001, US-002, ...
   - Format: 'As a [actor], I want to [action] so that [benefit].'
   - Priority: High / Medium / Low
   - Story Points: Fibonacci scale (1, 2, 3, 5, 8, 13)
   - Acceptance criteria (2-3 bullet points per story)
2. Sprint 1 Backlog — highest-priority stories that fit a 2-week sprint
3. Sprint Goal for Sprint 1 (one clear sentence)
4. Total Story Points across the full backlog
5. Story Points committed to Sprint 1
6. Assumptions and open questions

Important:
- Include actors: Farmer, Restaurant Buyer, Grocery Buyer, Logistics Coordinator, Platform Admin
- Include Cropchain-specific features: yield prediction, fair pricing, traceability, alerts, logistics visibility
- Use Fibonacci Story Points only
- Make the output structured and presentation-ready
""".strip()

# ── Step 2 prompt: Product Owner → Scrum Master ───────────────────────────────
# (This is filled dynamically after Step 1 runs and we extract the PO output)
# The placeholder below is used if you want to preview the prompt structure.
po_to_sm_prompt_template = """
The Product Owner has produced the following Product Backlog for Cropchain.

=== PRODUCT BACKLOG ===
{product_backlog_output}

Using this backlog, produce a detailed Sprint 1 Plan for the Cropchain Scrum team.

Your output must include:
1. Sprint 1 overview and duration (2 weeks)
2. Team capacity in Story Points (assume a team of 5 developers, 8 SP per person per sprint = 40 SP capacity)
3. Sprint 1 Backlog with story assignments to team roles
4. Sprint ceremonies schedule:
   - Sprint Planning (Day 1)
   - Daily Scrum (each day, 15 minutes)
   - Sprint Review (Day 10)
   - Sprint Retrospective (Day 10, after review)
5. Definition of Done (DoD) — at least 5 criteria
6. Identified risks and impediments for Sprint 1
7. Sprint 1 success criteria
8. Velocity assumption for future sprint planning
9. Assumptions and open questions

Make the output structured and presentation-ready.
""".strip()

print("Phase 2 Experiment 1 prompts prepared.")
print("Step 1 prompt preview (first 500 chars):")
print(customer_to_po_prompt[:500] + "\n...")

## Section 5: Output Persistence Helpers

These helper functions save Phase 2 outputs for:
- Teammate reuse in Experiment 2 (Developer + QA agents)
- Teammate reuse in Experiment 3 (DevOps + TechWriter agents)
- Final comparative analysis across Waterfall vs Scrum
- Presentation and demo preparation

In [ ]:
def save_text(filename: str, text: str):
    """Save a text string to a file in the output directory."""
    file_path = OUTPUT_DIR / filename
    file_path.write_text(text, encoding="utf-8")
    print(f"Saved: {file_path}")

def save_chat_history(filename: str, chat_result):
    """Save the full chat history from an AutoGen chat result."""
    file_path = OUTPUT_DIR / filename
    if hasattr(chat_result, "chat_history"):
        lines = []
        for item in chat_result.chat_history:
            role = item.get("name", item.get("role", "Unknown"))
            content = item.get("content", "")
            lines.append(f"{role}:\n{content}\n")
        file_path.write_text(("\n" + ("-" * 80) + "\n").join(lines), encoding="utf-8")
        print(f"Saved chat history: {file_path}")
    else:
        print("No chat_history found on chat result.")

def extract_last_message(chat_result) -> str:
    """Extract the last assistant message from a chat result."""
    if hasattr(chat_result, "chat_history") and chat_result.chat_history:
        return chat_result.chat_history[-1].get("content", "")
    return ""

print("Output helpers defined.")

## Section 6: Experiment 1 Objective and Focus Areas

### Objective
Transform the Cropchain customer problem into a Scrum Product Backlog and Sprint 1 Plan.

### Focus Areas
- Customer problem interpretation through a Scrum lens
- Product Backlog creation with prioritized user stories
- Story Point estimation using the Fibonacci scale
- Sprint 1 planning with capacity, ceremonies, and Definition of Done
- Baseline artifact creation for teammate notebooks

### Why This Experiment Matters
The Product Backlog and Sprint 1 Plan produced here serve as the foundation for:
- Teammate Experiment 2: Sprint execution, development tasks, QA coverage
- Teammate Experiment 3: Sprint Review, retrospective, documentation, and Scrum vs Waterfall analysis

## Section 7: Run Experiment 1

This section executes the two-step workflow:

**Step 1:** Customer Proxy → Product Owner (generates Product Backlog)  
**Step 2:** Product Owner → Scrum Master (generates Sprint 1 Plan)

In [ ]:
def run_phase2_experiment_1():

    # ── STEP 1: Customer Proxy → Product Owner ────────────────────────────────
    print("=" * 80)
    print("STEP 1: Customer Proxy → Product Owner")
    print("=" * 80)

    customer_to_po_result = customer_proxy_agent.initiate_chat(
        recipient=product_owner_agent,
        message=customer_to_po_prompt,
        max_turns=1,
    )

    # Extract PO output for Step 2
    product_backlog_output = extract_last_message(customer_to_po_result)

    # ── STEP 2: Product Owner → Scrum Master ──────────────────────────────────
    print("\n" + "=" * 80)
    print("STEP 2: Product Owner → Scrum Master")
    print("=" * 80)

    po_to_sm_prompt = po_to_sm_prompt_template.format(
        product_backlog_output=product_backlog_output
    )

    po_to_sm_result = product_owner_agent.initiate_chat(
        recipient=scrum_master_agent,
        message=po_to_sm_prompt,
        max_turns=1,
    )

    return customer_to_po_result, po_to_sm_result

# Run the experiment
customer_to_po_result, po_to_sm_result = run_phase2_experiment_1()

## Section 8: Save Experiment 1 Artifacts

All outputs are saved to `src/outputs/phase2/experiment_1/`.  
Teammates **must load** files `03_product_backlog.md` and `04_sprint1_plan.md` as inputs to their notebooks.

In [ ]:
# ── Save raw chat histories ───────────────────────────────────────────────────
save_chat_history("01_customer_to_po_chat.txt", customer_to_po_result)
save_chat_history("02_po_to_sm_chat.txt", po_to_sm_result)

# ── Save clean markdown artifacts for teammate reuse ─────────────────────────
product_backlog_text = extract_last_message(customer_to_po_result)
sprint1_plan_text    = extract_last_message(po_to_sm_result)

if product_backlog_text:
    save_text("03_product_backlog.md", product_backlog_text)

if sprint1_plan_text:
    save_text("04_sprint1_plan.md", sprint1_plan_text)

print("\nPhase 2 Experiment 1 artifacts saved.")

## Section 9: Artifact Validation

In [ ]:
required_files = [
    OUTPUT_DIR / "03_product_backlog.md",
    OUTPUT_DIR / "04_sprint1_plan.md",
]

all_found = True
for f in required_files:
    status = "FOUND" if f.exists() else "MISSING"
    print(f"{f.name}: {status}")
    if status == "MISSING":
        all_found = False

if all_found:
    print("\nAll Phase 2 Experiment 1 artifacts validated successfully.")
    print("Teammates can now load these files in their notebooks.")
else:
    print("\nWARNING: Some files are missing. Re-run Section 7 before notifying teammates.")

## Section 10: Display Generated Outputs

In [ ]:
from IPython.display import Markdown, display

backlog_path = OUTPUT_DIR / "03_product_backlog.md"
sprint_path  = OUTPUT_DIR / "04_sprint1_plan.md"

if backlog_path.exists():
    print("=" * 60)
    print("PRODUCT BACKLOG OUTPUT")
    print("=" * 60)
    display(Markdown(backlog_path.read_text(encoding="utf-8")))

if sprint_path.exists():
    print("=" * 60)
    print("SPRINT 1 PLAN OUTPUT")
    print("=" * 60)
    display(Markdown(sprint_path.read_text(encoding="utf-8")))

## Section 11: Experiment 1 Evaluation

### What This Experiment Produced
- A Scrum-aligned Product Backlog with prioritized user stories
- Fibonacci Story Point estimates for each story
- Acceptance criteria per user story
- A Sprint 1 Plan with team capacity, ceremonies, and Definition of Done
- Reusable markdown artifacts for teammate notebooks

### Quality Assessment

**Accuracy:** The Product Backlog directly traces to the 17 Phase 1 requirements (REQ-001 through REQ-017), ensuring continuity between Waterfall planning and Scrum execution.

**Coherence:** The Sprint 1 Plan is derived directly from the Product Backlog, not generated independently. The Scrum Master did not redefine requirements.

**Improvements over Phase 1:**
- All agents active from the start (no commented-out code)
- Consistent Fibonacci Story Point estimation (no SLOC/Feature Point inconsistency)
- Explicit Sprint ceremony schedule
- Definition of Done included
- Phase 1 requirements injected as context for continuity

### Limitations
- Story Point estimates are LLM-generated approximations, not real team velocity
- Sprint capacity assumes a fixed team size (5 developers × 8 SP = 40 SP/sprint)
- Actual velocity will be measured after Sprint 1 Review

### Conclusion
Phase 2 Experiment 1 successfully establishes the Scrum planning baseline for Cropchain.
The artifacts produced here are ready for teammate use in Experiments 2 and 3.

## Section 12: Git Branch Strategy

Run the following commands in your terminal to set up the three-branch structure.

**Branch layout:**
- `main` — original Phase 1 code (preserved, untouched)
- `phase1-backup` — safety copy of Phase 1 
- `phase2` — active working branch for Phase 2

```bash
# 1. Make sure you are on main and everything is committed
git checkout main
git status   # should be clean

# 2. Create a safety backup of Phase 1
git checkout -b phase1-backup
git push -u origin phase1-backup

# 3. Go back to main and create the Phase 2 working branch
git checkout main
git checkout -b phase2
git push -u origin phase2

# 4. From now on, all Phase 2 work is committed to the phase2 branch
# Add your Phase 2 notebooks:
git add 04_Cropchain_Phase2_Setup_and_Experiment1_Scrum.ipynb
git commit -m "Phase 2 Exp 1: Team Lead - Scrum setup, Product Backlog, Sprint 1 Plan"
git push origin phase2
```

> **Result:** You will have 3 branches total — `main`, `phase1-backup`, `phase2`.  
> `main` and `phase1-backup` are both safe copies of Phase 1.  
> `phase2` is where all Phase 2 development happens.